# Clases

Hasta ahora se han visto los tipos de datos básicos de Python: cadenas (de texto), números, listas, tuplas, conjuntos y diccionarios. Las **clases** permiten definir tipos de datos propios que combinan información (*atributos*) y comportamiento (*métodos*), para modelar cualquier elemento de interés: una estación meteorológica, una parcela catastral o una capa ráster. Las bibliotecas de análisis geoespacial (GeoPandas, Shapely, Rasterio, etc.) están construidas con clases, por lo que comprenderlas es imprescindible para utilizarlas con soltura.

Este capítulo introduce el vocabulario de la programación orientada a objetos (POO, OOP por sus siglas en inglés) y su implementación en Python. Se recomienda ejecutar cada ejemplo y modificarlo: los conceptos se afianzan con la práctica.

## ¿Qué son las clases?

Una clase es una **plantilla** que describe qué información almacena un tipo de elemento y qué acciones puede realizar.

Una analogía útil en un SIG es una capa cargada en un proyecto: toda capa tiene propiedades (nombre, sistema de referencia, extensión) y acciones (hacer zoom a la capa, cambiar su simbología, exportarla). La clase es la plantilla común y cada capa cargada es un objeto con sus propios valores. Lo esencial es que información y acciones van juntas: las acciones trabajan sobre la información del propio objeto.

El siguiente ejemplo define la clase `Location`, que representa un punto en un sistema de coordenadas proyectado (en metros). La versión inicial almacena una posición (x, y) y realiza una acción: desplazarse hacia el norte.

In [ ]:
class Location:
    """Represent a point in a projected coordinate system (metres)."""

    def __init__(self):
        # Each location has an (x, y) position: easting and northing.
        self.x = 0
        self.y = 0

    def move_north(self):
        """Increase the northing by 1 metre."""
        self.y += 1

Definir la clase no crea ningún punto: solo describe cómo serán. Para crear un **objeto** (también llamado **instancia**) se llama a la clase como si fuera una función. Los atributos y métodos del objeto se utilizan con la *notación de punto*: `my_location.y` accede al atributo `y` y `my_location.move_north()` ejecuta el método.

Al imprimir el objeto, Python muestra su clase y su dirección en memoria.

In [ ]:
my_location = Location()
print(my_location)
print(type(my_location))

print("Northing:", my_location.y)
my_location.move_north()
print("Northing:", my_location.y)
my_location.move_north()
print("Northing:", my_location.y)

Los tipos de datos vistos hasta ahora ya eran clases. `str` es la clase de los textos, y cada valor entre comillas es un **objeto** de esa clase que Python crea automáticamente.

In [ ]:
station_name = "Daroca"
print(type(station_name))
print(station_name.upper())

La función `type()` muestra la clase a la que pertenece un objeto, y `upper()` es un método de `str` que se utiliza con la misma notación de punto que `my_location.move_north()`.

La función `dir()` lista los nombres de los atributos y métodos de un objeto. Se aplica a un texto y al objeto `my_location`:

In [ ]:
print(dir(station_name))
print(dir(my_location))

Las listas son largas porque incluyen muchos nombres con doble guion bajo (`__`). Son métodos especiales que Python utiliza internamente y, por ahora, se pueden ignorar. `dir()` ordena los nombres alfabéticamente y el guion bajo se ordena antes que las minúsculas, así que los nombres útiles quedan al final: en `str`, métodos como `upper` o `lower`; en `my_location`, los atributos `x` e `y` y el método `move_north`.

Para ver solo los nombres útiles, se pueden filtrar con una comprensión de listas. El método `.startswith()` de `str` devuelve `True` si un texto empieza por el fragmento indicado. Después, `.join()` une los nombres en un único texto para imprimirlos como una lista:

In [ ]:
public_names = [name for name in dir(my_location) if not name.startswith("__")]
print(public_names)

names_text = "\n- ".join(public_names)
print(f"Location attributes and methods:\n- {names_text}")

Igual que se pueden crear tantos objetos `str` como se necesiten, también se pueden crear tantos objetos de una clase propia como se requieran. Cada uno tiene sus propios atributos, de modo que modificar uno no afecta a los demás:

In [ ]:
# Create five locations and store them in a list.
locations = []
for i in range(5):
    locations.append(Location())

# Move only the first one.
locations[0].move_north()

for location in locations:
    print("Northing:", location.y)

Solo ha cambiado la primera localización.

### Ejercicio 1: puntos sin clases

Sin utilizar clases, escribir un programa que:

1. Guarde 5 estaciones de medición en una lista de diccionarios, cada una con `name`, `x` e `y` (coordenadas proyectadas, en metros).
2. Defina una función `move_north(station)` que aumente en 1 la coordenada `y` de una estación.
3. Desplace la primera estación hacia el norte dos veces y muestre la coordenada `y` de todas ellas.

El objetivo es comprobar lo incómodo del enfoque, no profundizar en él.

In [ ]:
stations = [
    {"name": "Retiro", "x": 440000, "y": 4474000},
    {"name": "Cuatro Vientos", "x": 434000, "y": 4469000},
    {"name": "Barajas", "x": 450000, "y": 4480000},
    {"name": "Getafe", "x": 438000, "y": 4465000},
    {"name": "Vallecas", "x": 443000, "y": 4470000},
]

Con diccionarios se obtiene un resultado parecido, pero nada vincula la función `move_north()` a las estaciones: acepta cualquier diccionario, y su funcionamiento depende de que las claves (`"x"`, `"y"`) estén bien escritas en cada uno. Una clase, en cambio, agrupa datos y comportamiento en un único elemento: el método `move_north()` pertenece a `Location` y solo está disponible en sus objetos, que siempre tienen los atributos `x` e `y`.

## Terminología

- **Clase**: código que define los atributos y comportamientos necesarios para modelar algo, ya sea del mundo real (una estación meteorológica) o virtual (una capa en un proyecto SIG).
- **Atributo**: dato que forma parte de la clase; en código, una variable asociada al objeto.
- **Método**: función definida dentro de una clase; define un comportamiento.
- **Objeto** (o **instancia**): ejemplar concreto de una clase, con valores propios para cada atributo. Una clase admite tantos objetos como se quiera.

Estos términos cobrarán sentido con más ejemplos.

## Análisis de la clase `Location`

**Definición.** La palabra clave `class` indica que se va a definir una clase. Se aplican las mismas reglas de nombres que a las variables, pero por convención los nombres de clase se escriben en *CamelCase*: cada palabra empieza por mayúscula y sin guiones bajos (`Location`, `RasterLayer`). Justo debajo de la definición se escribe un *docstring* que resuma la finalidad de la clase, igual que en las funciones (cuaderno `07_functions.ipynb`).

**El método `__init__()`.** Los nombres que empiezan y terminan con doble guion bajo son métodos especiales que Python utiliza en situaciones concretas. `__init__()` se ejecuta automáticamente al crear un objeto y sirve para asignar valores iniciales a sus atributos antes de usarlo. `__init__` es uno de los nombres con doble guion bajo que aparecían en la salida de `dir()`.

**Los métodos.** Un método es una función que pertenece a una clase, por lo que admite todo lo visto con las funciones: argumentos posicionales, con nombre, valores por defecto y valores devueltos con `return`.

**El parámetro `self`.** Es lo más difícil de entender al principio. `self` es una referencia al objeto concreto que llama al método, y permite acceder a sus atributos. Todo método debe recibirlo como primer parámetro. Al escribir `my_location.move_north()`, Python lo traduce internamente a `Location.move_north(my_location)`: el objeto `my_location` pasa a ser `self`. Ambas formas son equivalentes:

In [ ]:
my_location = Location()
my_location.move_north()
Location.move_north(my_location)
print(my_location.y)

Dentro del método, `self.y += 1` modifica el atributo `y` del objeto que lo ha llamado y de ningún otro. Por eso cada objeto es independiente de los demás.

## Perfeccionar la clase `Location`

### Parámetros en `__init__()`

Con parámetros con valor por defecto, cada objeto puede crearse en cualquier posición. Si no se indican, se usa el valor por defecto (0, 0):

In [ ]:
class Location:
    """Represent a point in a projected coordinate system (metres)."""

    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

In [ ]:
# Coordinates in ETRS89 / UTM zone 30N (EPSG:25830), in metres.
locations = [
    Location(),
    Location(440000, 4474000),
    Location(x=441000, y=4475000),
]

for index, location in enumerate(locations):
    print(f"Location {index} is at ({location.x}, {location.y}).")

### Parámetros en un método

Cualquier método puede recibir parámetros. El método `move_north()` puede generalizarse como `move()`, que desplaza la localización una cantidad cualquiera en cualquier dirección. Sus parámetros se llaman `dx` y `dy` (y no `x` e `y`) para dejar claro que son *incrementos*, no nuevas posiciones. Los valores por defecto definen un comportamiento razonable: sin argumentos, el punto se desplaza 1 m al norte. Los valores negativos desplazan hacia el oeste (`dx`) o hacia el sur (`dy`).

In [ ]:
class Location:
    """Represent a point in a projected coordinate system (metres)."""

    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def move(self, dx=0, dy=1):
        """Shift the location by (dx, dy). Default: 1 metre north."""
        self.x += dx
        self.y += dy

In [ ]:
help(Location)

In [ ]:
location = Location(440000, 4474000)
location.move()
location.move(10, 10)
location.move(-30, 0)
print(f"Location is at ({location.x}, {location.y}).")

### Un método que devuelve un valor

Un método también puede recibir otro objeto de la misma clase. El método `get_distance()` calcula la distancia euclídea entre la localización que lo llama (`self`) y otra (`other`), y la devuelve con `return`.

> **Atención**: esta fórmula solo es válida con coordenadas *proyectadas* (en metros). Con latitud y longitud en grados daría resultados incorrectos; en ese caso se necesita la fórmula de Haversine o una biblioteca como `pyproj`.

In [ ]:
from math import sqrt


class Location:
    """Represent a point in a projected coordinate system (metres)."""

    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def move(self, dx=0, dy=1):
        """Shift the location by (dx, dy). Default: 1 metre north."""
        self.x += dx
        self.y += dy

    def get_distance(self, other):
        """Return the planar distance (metres) to another location."""
        return sqrt((self.x - other.x)**2 + (self.y - other.y)**2)

In [ ]:
zaragoza = Location(675032.62, 4612903.79)
belchite = Location(688277.33, 4575146.69)
print(f"The locations are {zaragoza.get_distance(belchite):.2f} m apart.")

Una clase puede ampliarse con tantos atributos y métodos como requiera el problema (nombre, sistema de referencia, altitud, operaciones espaciales, etc.).

### Ejercicios

#### Ejercicio 2: Recrear `Location`

Sin consultar los ejemplos, escribir la clase `Location` con `__init__()`:

- atributos `x` e `y`, por defecto 0
- métodos `move()` y `get_distance()`.

Crear varias localizaciones, moverlas y mostrar las distancias entre ellas, para comprobar que cada objeto es independiente.

#### Ejercicio 3: Atributos

Partiendo de `Location`, añadir a `__init__()` dos atributos nuevos:

- `name`, obligatorio (sin valor por defecto). Antes de guardarlo:
    - Si es de tipo `str`, eliminar los espacios del principio y del final y convertirlo a minúsculas.
    - Si no es de tipo `str`, asignar `None` al atributo e imprimir un aviso.
- `crs_code`, con valor por defecto `25830`.

*Pistas:* `__init__()` no devuelve ningún valor, solo asigna atributos. Los parámetros obligatorios deben escribirse antes que los que tienen valor por defecto.

Crear varias localizaciones, al menos una con un nombre que no sea texto, e imprimir los atributos de cada una.

#### Ejercicio 4: Métodos

Añadir a `Location` los siguientes métodos y probarlos:

- `is_inside(min_x, min_y, max_x, max_y)`: devuelve `True` si la localización está dentro de un rectángulo delimitador (*bounding box*).
- `warn_if_close(other, threshold)`: imprime un aviso si la distancia a otra localización es menor que `threshold`. Una distancia de 0 puede deberse a comparar una localización consigo misma.

#### Ejercicio 5: Clase `Parcel`

Modelar una parcela catastral:

- atributos `parcel_id`, `owner`, `area_m2` y `land_use`
- método `describe()` (imprime una ficha)
- método `change_land_use(new_use)`
- método `area_ha()` (devuelve la superficie en hectáreas) *Pista: 1 ha = 10000 m2*

Crear varias parcelas y utilizar los métodos.

#### Ejercicio 6: Clase `RasterLayer`

- Atributos `name`, `crs_code`, `cell_size`, `n_rows` y `n_cols`
- método `get_cell_count()` *Pista: multiplicar sus dimensiones*
- método `get_area()`, en m2, número de celdas por `cell_size` al cuadrado
- método `describe()`, mostrar un informe de las características de la capa, incluyendo su área y el número de celdas.

Crear varias capas con resoluciones distintas y comparar sus superficies.

## Herencia

Un objetivo central de la POO es escribir código fiable y reutilizable. Si hubiera que crear una clase nueva desde cero para cada tipo de elemento, apenas habría código reutilizable. La **herencia** resuelve este problema: una clase puede basarse en otra y heredar todos sus atributos y métodos.

- La clase original se denomina clase **padre** o **superclase**; la nueva, clase **hija** o **subclase**.
- La clase hija puede añadir atributos y métodos propios, que la clase padre no posee.
- La clase hija puede **sobrescribir** (*override*) un método del padre: si define un método con el mismo nombre, sus objetos usarán el nuevo en lugar del heredado.

Una estación de medición es un caso particular de localización: tiene una posición, pero además un nombre y un número de lecturas registradas. En lugar de escribir una clase nueva, `Station` hereda de `Location` y añade lo que le diferencia. El nombre de la clase padre se escribe entre paréntesis:

```python
class ChildClass(ParentClass):
```

El `__init__()` de la clase hija debe recibir los parámetros de la clase padre y pasárselos al `__init__()` de esta, lo que se hace con `super().__init__()`. Aquí se añade también `describe()` a `Location`, y `Station` lo sobrescribe.

In [ ]:
from math import sqrt


class Location:
    """Represent a point in a projected coordinate system (metres)."""

    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def move(self, dx=0, dy=1):
        self.x += dx
        self.y += dy

    def get_distance(self, other):
        return sqrt((self.x - other.x)**2 + (self.y - other.y)**2)

    def describe(self):
        print(f"Location at ({self.x}, {self.y}).")


class Station(Location):
    """A measurement station: a location with a name and a reading counter."""

    def __init__(self, x=0, y=0, name="Unnamed", readings_count=0):
        super().__init__(x, y)
        self.name = name
        self.readings_count = readings_count

    def describe(self):
        # Overrides Location.describe().
        print(f"Station {self.name} at ({self.x}, {self.y}), "
              f"{self.readings_count} readings.")

In [ ]:
belchite = Location(688277.33, 4575146.69)
station = Station(440000, 4474000, "Retiro", 12)

# Each object uses its own version of describe().
for item in (belchite, station):
    item.describe()

# Station also inherits move() and get_distance() from Location.
print(f"Distance to Belchite {station.get_distance(belchite) / 1000:.2f} km")

`Station` conserva toda la funcionalidad de `Location` (puede moverse y calcular distancias, incluso a objetos de la clase padre) y añade la suya.

Al definir una clase hija, se puede llamar al `__init__()` del padre nombrándolo de forma explícita y pasando `self` a mano:

```python
Location.__init__(self, x, y)
```

Lo habitual, sin embargo, es usar `super()`:

```python
super().__init__(x, y)
```

Así no hace falta repetir el nombre del padre, y el código sigue funcionando si más adelante se cambia la clase base.

La llamada explícita solo es necesaria en casos avanzados, como la herencia múltiple cuando se quiere inicializar un padre concreto.

### Ejercicios

#### Ejercicio 7: Herencia con `Parcel`

Partiendo del ejercicio 5, crear la clase `AgriculturalParcel`, hija de `Parcel`, con los atributos `crop` (tipo de cultivo, como texto) e `irrigated` (booleano: `True` si la parcela es de regadío), y sobrescribir `describe()` para que muestre también estos datos reutilizando `super().describe()`.

Crear un objeto y comprobar que funcionan atributos y métodos de ambas clases.

#### Ejercicio 8: Refinar `Station`

Añadir los atributos `elevation` y `sensor_types` (una lista) y un método `add_reading()` que incremente `readings_count`. Comprobar que funcionan creando una estación y llamando al método varias veces.

## Módulos propios

En el cuaderno `08_import_open.ipynb` se importaron módulos de la biblioteca estándar. También se pueden crear módulos propios: basta con guardar clases y funciones en un archivo `.py`. A medida que los programas crecen, esto permite dividir el código en archivos más manejables y reutilizarlo en varios programas.

La PEP 8 recomienda que los [nombres de los módulos](https://peps.python.org/pep-0008/#package-and-module-names) sean cortos y estén en minúsculas, con guiones bajos para separar palabras si es necesario (`location.py`, `raster_layer.py`). Los [nombres de las clases](https://peps.python.org/pep-0008/#class-names), como se ha visto, se escriben en CamelCase y sin guiones bajos. Esta diferencia permite distinguir a simple vista módulos y clases, por ejemplo, en una importación como `from location import Location`: `location` es el módulo y `Location`, la clase.

A continuación se guardan `Location` y `Station` en el módulo `location.py`. La primera línea (`%%writefile`) es una instrucción de Jupyter que escribe el contenido de la celda en ese archivo, en la carpeta del cuaderno.

In [ ]:
%%writefile location.py
from math import sqrt


class Location:
    """Represent a point in a projected coordinate system (metres)."""

    def __init__(self, x=0, y=0):
        self.x = x
        self.y = y

    def move(self, dx=0, dy=1):
        self.x += dx
        self.y += dy

    def get_distance(self, other):
        return sqrt((self.x - other.x)**2 + (self.y - other.y)**2)

    def describe(self):
        print(f"Location at ({self.x}, {self.y}).")


class Station(Location):
    """A measurement station: a location with a name and a reading counter."""

    def __init__(self, x=0, y=0, name="Unnamed", readings_count=0):
        super().__init__(x, y)
        self.name = name
        self.readings_count = readings_count

    def describe(self):
        print(f"Station {self.name} at ({self.x}, {self.y}), "
              f"{self.readings_count} readings.")

Un programa que utilice el módulo queda limpio y sin los detalles de implementación. Python busca `location.py` primero en la carpeta del programa (en Jupyter, la del cuaderno) y después en la biblioteca estándar y en los paquetes instalados. Por eso, un módulo propio no debe llamarse igual que uno existente, como `math.py`.

In [ ]:
from location import Location, Station

location = Location()
location.describe()

station = Station(name="Retiro")
station.describe()

Aunque solo se importen algunas clases, Python ejecuta el módulo completo; elegir qué importar sirve para controlar los nombres que quedan disponibles en el programa, no para ahorrar tiempo de procesamiento.

### Formas de importar

Las formas de importar vistas en el cuaderno `08_import_open.ipynb` funcionan igual con los módulos propios.

Con `import location`, las clases se usan con notación de punto (`location.Location`) y el nombre `location` pasa a designar el **módulo**. Si después se asignara un valor a una variable llamada `location`, ese nombre dejaría de referirse al módulo y sus clases ya no serían accesibles a través de él. Por eso, en el siguiente ejemplo, las variables tienen otros nombres:

In [ ]:
import location

origin = location.Location()
retiro = location.Station(name="Retiro")
retiro.describe()

Un alias evita este conflicto, porque el módulo queda disponible con otro nombre. Conviene que el alias sea descriptivo: `location_module` es más legible que `lm`.

In [ ]:
import location as location_module

origin = location_module.Location()
retiro = location_module.Station(name="Retiro")
retiro.describe()

También existe la forma `from module_name import *`, que importa todos los nombres del módulo. Conviene evitarla: se desconoce qué nombres entran en el programa y es fácil que alguno coincida con una variable propia. Si se necesita todo el módulo, es preferible `import module_name`.

### Un módulo de funciones

Los módulos también pueden contener funciones que no pertenecen a ninguna clase. Por ejemplo, un módulo `conversions.py` con conversiones de unidades habituales en SIG:

In [ ]:
%%writefile conversions.py
def m_to_km(distance):
    """Convert metres to kilometres."""
    return distance / 1000


def ha_to_m2(area):
    """Convert hectares to square metres."""
    return area * 10000


def m_to_ft(distance):
    """Convert metres to feet."""
    return distance * 3.28084

Las funciones se importan igual que las clases:

In [ ]:
from conversions import m_to_km
import conversions
import conversions as conv

print(m_to_km(2500))
print(conversions.ha_to_m2(3))
print(conv.m_to_ft(100))

### Ejercicios

#### Ejercicio 9: Importar `Parcel`

Guardar `Parcel` y `AgriculturalParcel` (ejercicios 5 y 7) en un módulo `parcel.py`.

Escribir el código de ejemplo que las utilice llamando al módulo utilizando la sintaxis que se considere más apropiada.

#### Ejercicio 10: Importar `RasterLayer`

Guardar la clase del ejercicio 6 en `raster_layer.py`. Importar la clase de una forma diferente a la del ejercicio anterior y crear un código de ejemplo que la utilice.

#### Ejercicio 11: Eliminar archivos con `Path`

Si se han ejecutado las celdas anteriores y realizado los ejercicios 9 y 10, en la carpeta de los cuadernos aparecen los archivos `location.py`, `conversions.py`, `parcel.py` y `raster_layer.py`: los módulos creados en este cuaderno.

Utilizar `Path` y su método `.unlink()`, que elimina el archivo al que apunta la ruta, para borrarlos.

*Pista:* `Path()` admite varios textos como argumentos y los une como partes de una ruta: `Path("folder", "file.py")` equivale a `Path("folder") / "file.py"`. Se puede comprobar con `help(Path)`.

<a name="mro"></a>
## Orden de resolución de métodos (MRO) (opcional)

> **Sección opcional.** Trata un caso avanzado que no es necesario para el resto del curso.

Una clase puede heredar de varias clases a la vez (*herencia múltiple*). Cuando se llama a un método, Python lo busca en una lista ordenada de clases denominada **MRO** (*method resolution order*), de izquierda a derecha, y usa la primera coincidencia. La lista puede consultarse con `mro()`.

En el ejemplo, `Landmark` (un hito geográfico) hereda de `Located` (posición) y `Named` (nombre). Como cada `__init__()` recibe argumentos distintos, se llaman de forma explícita. El método `describe()` solo existe en `Describable`, de la que hereda `Named`, y Python lo encuentra recorriendo el MRO.

El ejemplo utiliza dos elementos nuevos:

- `self.__class__` es la clase del objeto y `.__name__` es su nombre como texto. Así, `describe()` muestra `Landmark` aunque esté definido en `Describable`.
- `super()` con dos argumentos, `super(Landmark, landmark)`, busca el método en las clases que aparecen **después** de `Landmark` en el MRO del objeto `landmark`. Dentro de un método, `super()` sin argumentos equivale a `super(CurrentClass, self)`. Aquí ambas llamadas dan el mismo resultado.

In [ ]:
class Located:
    def __init__(self, x, y):
        self.x = x
        self.y = y


class Describable:
    def describe(self):
        print("Description from type:", self.__class__.__name__)


class Named(Describable):
    def __init__(self, name):
        self.name = name


class Landmark(Located, Named):
    def __init__(self, x, y, name):
        Located.__init__(self, x, y)
        Named.__init__(self, name)


print("MRO:", Landmark.mro())

landmark = Landmark(440000, 4474000, "Puerta de Alcalá")
print(landmark.x, landmark.y, landmark.name)

landmark.describe()
super(Landmark, landmark).describe()

Con herencia simple, el MRO es trivial (la clase, su padre, el padre de este, y así hasta `object`). Solo en la herencia múltiple conviene consultarlo para saber qué versión de un método se ejecutará.

## Resumen

- Una **clase** es una plantilla que agrupa información (**atributos**) y acciones (**métodos**). Un **objeto** es un ejemplar concreto de una clase, con sus propios valores. Los tipos vistos hasta ahora, como `str`, ya eran clases.
- `__init__()` se ejecuta al crear un objeto y fija sus atributos iniciales. `self` es el propio objeto que llama al método, y todo método lo recibe como primer parámetro.
- Los atributos y métodos se utilizan con **notación de punto** (`objeto.atributo`, `objeto.metodo()`). `type()` muestra la clase de un objeto y `dir()` lista sus atributos y métodos.
- Los parámetros con valores por defecto hacen que los métodos y `__init__()` sean más flexibles, y un método puede devolver valores con `return`.
- La **herencia** permite crear una clase hija que reutiliza una clase padre, añade elementos propios y puede sobrescribir métodos. `super()` da acceso a la clase padre. Con herencia múltiple, el **MRO** indica el orden en que Python busca los métodos.
- Las clases se guardan en **módulos** (archivos `.py`) que se importan con `from ... import`, `import` o `import ... as`. Se debe evitar `import *`.

Las clases resultan especialmente útiles en proyectos que crecen, donde varios elementos comparten datos y comportamiento. En cambio, si un programa solo debe realizar una tarea concreta, como recortar un ráster con una máscara, definir una clase suele ser innecesario. En ese caso bastan unas pocas funciones. Conviene **recurrir a las clases cuando aportan claridad**, no por defecto.